In [4]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder \
    .appName("DataCleaning") \
    .getOrCreate()

# Optional: reduce log verbosity for cleaner output
spark.sparkContext.setLogLevel("WARN")

In [ ]:
data_file = "rea_data/domain/Data/vic_rentals_all.csv"
sdf = spark.read.csv(data_file, header=True, inferSchema=True)

26/09/11 10:51:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+----------------+--------+-----------+------+--------------------+-----------+-----------+--------+---------+---------+--------------------+--------------------+----------+---------+-------------------+--------------+-----------+-----------+-----------+----------------+------------+---------------+--------------------+--------------------+---------+--------------------+--------------------+--------------------+---------+
|listing_id|          suburb|postcode|weekly_rent|  bond|      available_date|date_listed|days_listed|bedrooms|bathrooms|carspaces|       property_type|             address|       lat|      lon|       scraped_date|domain_page_id|property_id|photo_count|video_count|floorplans_count|virtual_tour|   primary_type|      secondary_type|              agency|agency_id|         agent_names| structured_features|                 url|land_area|
+----------+----------------+--------+-----------+------+--------------------+-----------+-----------+--------+---------+-----

In [13]:
cols_drop = ["listing_id", "property_id", "domain_page_id", "url", 
             "scraped_date", "photo_count", "video_count", "floorplans_count",
             "virtual_tour", "agency", "agency_id", "agent_names", "bond",
             "land_area"]

sdf = sdf.drop(*cols_drop)

In [14]:
for column in sdf.columns:
    missing_val = sdf.filter(F.col(column).isNull()).count()
    print(f"{column}: {missing_val}")

suburb: 14
postcode: 20
weekly_rent: 299
available_date: 157
date_listed: 25
days_listed: 25
bedrooms: 146
bathrooms: 72
carspaces: 1824
property_type: 21
address: 125
lat: 25
lon: 25
primary_type: 25
secondary_type: 25
structured_features: 1347


In [16]:
missing_location = sdf.filter(F.col("suburb").isNull() | F.col("postcode").isNull())

missing_location.select("suburb", "postcode", "address").show(50, truncate=False)

+------------------------------------------------------------------------------+--------+-------+
|suburb                                                                        |postcode|address|
+------------------------------------------------------------------------------+--------+-------+
|NULL                                                                          |NULL    |NULL   |
|https://www.domain.com.au/312-673-la-trobe-street-docklands-vic-3008-15802858 |NULL    |NULL   |
|NULL                                                                          |NULL    |NULL   |
|https://www.domain.com.au/312-673-la-trobe-street-docklands-vic-3008-17747866 |NULL    |NULL   |
|NULL                                                                          |NULL    |NULL   |
|NULL                                                                          |NULL    |NULL   |
|NULL                                                                          |NULL    |NULL   |
|https://www.domain.

In [18]:
sdf.select([F.min(c).alias(f"{c}_min") for c in sdf.columns] + 
           [F.max(c).alias(f"{c}_max") for c in sdf.columns]).show(truncate=False)

+-------------------------------------------------------------------------------+-------------------------+---------------+---------------------+---------------+---------------+------------+-------------+-------------+--------------------+-----------------------+---------+---------+----------------+--------------------+-----------------------+-----------------------------------------------------------------------+------------+---------------------------------------------------------------------------+-----------------------+---------------+---------------+------------+-------------+-------------+-----------------+----------------------+---------+--------+----------------+------------------+--------------------------------+
|suburb_min                                                                     |postcode_min             |weekly_rent_min|available_date_min   |date_listed_min|days_listed_min|bedrooms_min|bathrooms_min|carspaces_min|property_type_min   |address_min            |lat_

In [6]:
sdf.count()

12738